# Stop classification — Stage A: real railway stations vs. urban transit, from raw OSM

`classify_stops.ipynb` (the sibling notebook) works from a pre-filtered CSV export that only has name/coordinates/country — no OSM tags. That's enough for Stage B matching, but Stage A (heavy rail vs. subway/tram) needs the actual tags (`train`, `uic_ref`, `station`, `subway`, `tram`, `light_rail`), which only a raw OSM extract carries. This notebook reads that raw extract directly via `pyosmium` and produces the `station_mode` classification described in `STOP_CLASSIFICATION.md` §3 Stage A.

**Output of this notebook feeds Stage B**: swap `classify_stops.ipynb`'s `bahnhoefe_stops_sorted.csv` input for `data/osm_stations_classified.csv` produced here, and it gains `station_mode` for free.

## 0. Prerequisite: shrink the extract first

Run this once, outside the notebook, on the country `.osm.pbf` from Geofabrik (see `STOP_CLASSIFICATION.md` §2.1). It cuts a multi-GB country file down to a few MB containing only station-like objects, which is what this notebook actually reads:

```bash
osmium tags-filter germany-latest.osm.pbf \
    nwr/railway=station nwr/railway=halt nwr/public_transport=station \
    -o data/raw/stations_de.osm.pbf
```

Repeat per country and either process them one at a time (loop below) or merge the filtered extracts first with `osmium merge` if a single combined run is preferred.

In [1]:
from pathlib import Path

import osmium
import pandas as pd

RAW_OSM_DIR = Path("data/raw")  # one *.osm.pbf per country, output of the osmium tags-filter step above

## 1. Extract station objects from the raw extract

Stations can be OSM **nodes** (a single point — the common case) or **ways** (an area, e.g. a station building outline). We handle both: for ways, the first valid member node is used as an approximate center, since we only need a representative point for the later geo-matching step, not the station footprint.

Only the tags Stage A's rules actually need are pulled out explicitly; the rest of the raw tag dict is kept too in case a rule needs tuning later.

In [2]:
class StationHandler(osmium.SimpleHandler):
    """Collects railway/public_transport station nodes and ways with their classification-relevant tags."""

    STATION_RAILWAY_VALUES = {"station", "halt"}

    def __init__(self):
        super().__init__()
        self.rows: list[dict] = []

    def _collect(self, osm_id: int, osm_type: str, lat: float, lon: float, tags) -> None:
        tags = dict(tags)
        railway = tags.get("railway")
        public_transport = tags.get("public_transport")

        # Defensive re-check even though the .osm.pbf was already filtered upstream by osmium tags-filter
        if railway not in self.STATION_RAILWAY_VALUES and public_transport != "station":
            return

        self.rows.append(
            {
                "osm_type": osm_type,
                "osm_id": osm_id,
                "name": tags.get("name"),
                "lat": lat,
                "lon": lon,
                "country": tags.get("addr:country"),
                "railway": railway,
                "public_transport": public_transport,
                "uic_ref": tags.get("uic_ref"),
                "train": tags.get("train"),
                "station": tags.get("station"),
                "subway": tags.get("subway"),
                "tram": tags.get("tram"),
                "light_rail": tags.get("light_rail"),
                "all_tags": tags,
            }
        )

    def node(self, n) -> None:
        if n.location.valid():
            self._collect(n.id, "node", n.location.lat, n.location.lon, n.tags)

    def way(self, w) -> None:
        try:
            first_located = next(nd for nd in w.nodes if nd.location.valid())
        except StopIteration:
            return  # no located member node available (rare, e.g. way references nodes outside the extract)
        self._collect(w.id, "way", first_located.location.lat, first_located.location.lon, w.tags)

In [3]:
extract_files = sorted(RAW_OSM_DIR.glob("*.osm.pbf")) or sorted(RAW_OSM_DIR.glob("*.osm"))
if not extract_files:
    raise FileNotFoundError(
        f"No .osm.pbf/.osm files found in {RAW_OSM_DIR}. Run the osmium tags-filter step from the "
        "prerequisites cell first, or point RAW_OSM_DIR at wherever those extracts were written."
    )

handler = StationHandler()
for extract_file in extract_files:
    handler.apply_file(str(extract_file))
    print(f"{extract_file.name}: {len(handler.rows)} station objects collected so far")

osm = pd.DataFrame(handler.rows)
print(f"\nTotal station objects: {len(osm)}")

FileNotFoundError: No .osm.pbf/.osm files found in data\raw. Run the osmium tags-filter step from the prerequisites cell first, or point RAW_OSM_DIR at wherever those extracts were written.

## 2. Stage A — classify heavy rail vs. urban transit

Direct implementation of `STOP_CLASSIFICATION.md` §3 Stage A. Read the two helper functions together with the doc's rule list — the code mirrors it rule-for-rule:

- `train=yes` or `uic_ref` present → heavy-rail evidence.
- `station=subway`/`light_rail`, or `subway=yes`/`tram=yes`/`light_rail=yes` → transit evidence.
- Both present → `mixed` (keep — big hubs tag rail and metro on one object).
- Only heavy-rail evidence → `heavy_rail`.
- Only transit evidence → `urban_transit` (drop from further evaluation).
- Neither, but `station=` has some other, unrecognized value (funicular, ferry, monorail, ...) → `undecided` — mark it, don't guess.
- Neither, and no `station=` tag at all → `heavy_rail` (no transit indicators present).

One easy-to-miss pitfall reused from `classify_stops.ipynb`: `bool(float("nan"))` is `True` in Python, so a naive `bool(row["uic_ref"])` on a pandas column silently treats *every* row missing that tag as if it had one. All presence checks below go through `pd.isna`/`pd.notna` explicitly.

In [ ]:
def _split_tag_values(value) -> set[str]:
    """OSM tags can hold semicolon-separated multi-values; a missing tag (NaN) becomes an empty set."""
    if pd.isna(value):
        return set()
    return {v.strip() for v in str(value).lower().split(";") if v.strip()}


KNOWN_TRANSIT_STATION_VALUES = {"subway", "light_rail", "tram"}


def classify_station_mode(row: pd.Series) -> tuple[str, str]:
    """Return (station_mode, mode_rule) per STOP_CLASSIFICATION.md Stage A."""
    station_tags = _split_tag_values(row["station"])
    is_known_transit = (
        bool(station_tags & KNOWN_TRANSIT_STATION_VALUES)
        or row["subway"] == "yes"
        or row["tram"] == "yes"
        or row["light_rail"] == "yes"
    )
    has_uic_ref = pd.notna(row["uic_ref"]) and str(row["uic_ref"]).strip() != ""
    has_heavy_rail_evidence = row["train"] == "yes" or has_uic_ref

    if has_heavy_rail_evidence and is_known_transit:
        return "mixed", "heavy_rail_evidence_plus_transit_tag"
    if has_heavy_rail_evidence:
        return "heavy_rail", "train_yes_or_uic_ref"
    if is_known_transit:
        return "urban_transit", "transit_tag_without_heavy_rail_evidence"
    if station_tags - KNOWN_TRANSIT_STATION_VALUES:
        return "undecided", "unrecognized_station_tag_value"
    return "heavy_rail", "no_transit_indicators_present"

In [ ]:
osm[["station_mode", "mode_rule"]] = osm.apply(lambda r: pd.Series(classify_station_mode(r)), axis=1)
osm["station_mode"].value_counts()

### Sanity check: undecided bucket size

Per the design doc, a large `undecided` bucket is a signal to add a track-proximity pass before trusting it — worth checking before moving on.

In [ ]:
undecided_share = (osm["station_mode"] == "undecided").mean()
print(f"{undecided_share:.1%} of station objects are undecided")
osm.loc[osm["station_mode"] == "undecided", ["osm_id", "name", "station"]].head(20)

## 3. Output — GTFS-style stops, ready for Stage B

Same base columns as `classify_stops.ipynb` expects (`stop_id`/`Name (ASCII)`/`Latitude`/`Longitude`/country/timezone-ish fields), plus the new `station_mode`/`mode_rule` columns. `urban_transit` rows are kept (not dropped) per the non-destructive principle — Stage B and Stage C can filter on `station_mode` as needed, but nothing is deleted here.

In [ ]:
result = osm[
    ["osm_id", "name", "lat", "lon", "country", "station_mode", "mode_rule", "uic_ref", "osm_type"]
].rename(
    columns={
        "osm_id": "stop_id",
        "name": "stop_name",
        "lat": "stop_lat",
        "lon": "stop_lon",
        "uic_ref": "stop_code",
    }
)
result["stop_id"] = "osm:" + result["osm_type"].str[0] + result["stop_id"].astype(str)  # e.g. osm:n123456
result = result.drop(columns="osm_type")

result.to_csv("data/osm_stations_classified.csv", index=False)
print(f"Wrote {len(result)} rows to data/osm_stations_classified.csv")
result.head()